In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('../../data/raw_data/heatpump/ECCC_2023.csv')
df = df[['AIRCOP', 'CLIENTPCODE', 'EGHSPACEENERGY', 'ERSSPACECOOLENERGY', 'FURNACEFUEL', 'FURSSEFF', 'PROVINCE', 'TYPEOFHOUSE', 'YEARBUILT']]


C:\Users\konar\AppData\Local\Temp\ipykernel_72344\4030634142.py:1: DtypeWarning: Columns (47,52,87,88,89,151,157,158,183,213,223,227,230,233,256,257,259,275,279,284,326,330,331,347,351,353,358) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('../../data/raw_data/heatpump/ECCC_2023.csv')


In [3]:
# find the number of missing values in each column
print(df.isnull().sum())

# # find the one missing PROVINCE value
#print(df[df['PROVINCE'].isnull()])

# # add QC TO THE MISSING VALUE
# df['PROVINCE'] = df['PROVINCE'].fillna('QC')

# # find the value counts of provicne
print(df['PROVINCE'].value_counts())


AIRCOP                1
CLIENTPCODE           0
EGHSPACEENERGY        0
ERSSPACECOOLENERGY    0
FURNACEFUEL           0
FURSSEFF              0
PROVINCE              0
TYPEOFHOUSE           0
YEARBUILT             0
dtype: int64
PROVINCE
ON    207867
QC    124488
AB     49022
BC     47556
NS     30470
NB     21605
PE      8801
NF      6469
MB      5844
SK      3733
YK       887
NT       181
Name: count, dtype: int64


In [4]:
# find the value counts of provicne
print(df['TYPEOFHOUSE'].value_counts())

# double/semi-detached, Row house, end unit, middle unit, attached duplex, attached triplex are single attached change to single attached
df['TYPEOFHOUSE'] = df['TYPEOFHOUSE'].replace(['Double/Semi-detached', 'Row house, end unit', 
                                               'Row house, middle unit', 'Attached Duplex', 'Attached Triplex'], 'Single Attached')
# appartment and appartment row are the same
df['TYPEOFHOUSE'] = df['TYPEOFHOUSE'].replace('Apartment Row', 'Apartment')

# find the value counts of provicne
print(df['TYPEOFHOUSE'].value_counts())

TYPEOFHOUSE
Single Detached           415107
Row house, end unit        24816
Double/Semi-detached       24136
Row house, middle unit     17779
Detached Duplex            14058
Attached Duplex             3235
Mobile Home                 2833
Apartment                   1424
Attached Triplex            1413
Detached Triplex            1296
Apartment Row                823
Single detached                1
Apartment (non-MURB)           1
Duplex (non-MURB)              1
Name: count, dtype: int64
TYPEOFHOUSE
Single Detached         415107
Single Attached          71379
Detached Duplex          14058
Mobile Home               2833
Apartment                 2247
Detached Triplex          1296
Single detached              1
Apartment (non-MURB)         1
Duplex (non-MURB)            1
Name: count, dtype: int64


In [5]:
# find the value counts of provicne
print(df['FURNACEFUEL'].value_counts())
# all the woods are the same
df['FURNACEFUEL'] = df['FURNACEFUEL'].replace(['Mixed Wood', 'Hardwood', 'Wood Pellets', 'Softwood'], 'Wood')
print(df['FURNACEFUEL'].value_counts())

FURNACEFUEL
Natural Gas     272490
Electricity     173623
Oil              47765
Propane           9483
Mixed Wood        3135
Hardwood           226
Wood Pellets       189
Softwood            12
Name: count, dtype: int64
FURNACEFUEL
Natural Gas    272490
Electricity    173623
Oil             47765
Propane          9483
Wood             3562
Name: count, dtype: int64


In [22]:
# find all values of PROVINCE
provinces = df['PROVINCE'].unique()
# make sub df with different values of PROVINCE
for prov in provinces:
    df_prov = df[df['PROVINCE'] == prov]
    # save to the csv file
    df_prov.to_csv(f'{prov}_2023.csv', index=False)

In [29]:
df['FURNACEFUEL'].unique()

array(['Natural Gas', 'Electricity', 'Oil', 'Propane', 'Wood'],
      dtype=object)

In [25]:
import os
def process_province(prov):
    df = pd.read_csv(prov + '_2023.csv')
    # get all values the TYPEOFHOUSE and YEARBUILT and FURNACEFUEL
    # types_of_house = df['TYPEOFHOUSE'].unique()
    
    # only consider single detached, single attached, appartment
    types_of_house = ['Single Detached', 'Single Attached', 'Apartment']
    furnace_fuels = df['FURNACEFUEL'].unique()
    years_built = [(0, 1945), (1946, 1960), (1961, 1977), (1978, 1983), (1984, 1995), (1996, 2000), (2001, 2005), (2006, 2010), (2011, 2015), (2016, 2020), (2021, 2025)]
    for types in types_of_house:
        for fuel in furnace_fuels:
            for year in years_built:
                df_sub = df[(df['TYPEOFHOUSE'] == types) & (df['FURNACEFUEL'] == fuel) & (df['YEARBUILT'] >= year[0]) & (df['YEARBUILT'] <= year[1])]
                # save to the csv file to QC folder
                if len(df_sub) > 1:
                    # if there is a '/' in the types, replace it with '-'
                    types = types.replace('/', '-')
                    directory = './' + prov
                    if not os.path.exists(directory):
                        os.makedirs(directory)
                    df_sub.to_csv(f'{directory}/{prov}_{types}_{fuel}_{year[0]}_{year[1]}_2023.csv', index=False)
                

In [14]:
!pip install scikit-learn

  Using cached joblib-1.4.2-py3-none-any.whl.metadata (5.4 kB)
  Using cached threadpoolctl-3.5.0-py3-none-any.whl.metadata (13 kB)
   ---------------------------------------- 0.0/11.1 MB ? eta -:--:--
    --------------------------------------- 0.3/11.1 MB ? eta -:--:--
   - -------------------------------------- 0.5/11.1 MB 2.1 MB/s eta 0:00:06
   --- ------------------------------------ 1.0/11.1 MB 1.7 MB/s eta 0:00:07
   --- ------------------------------------ 1.0/11.1 MB 1.7 MB/s eta 0:00:07
   ---- ----------------------------------- 1.3/11.1 MB 1.4 MB/s eta 0:00:08
   ----- ---------------------------------- 1.6/11.1 MB 1.4 MB/s eta 0:00:07
   ------ --------------------------------- 1.8/11.1 MB 1.4 MB/s eta 0:00:07
   ------- -------------------------------- 2.1/11.1 MB 1.3 MB/s eta 0:00:07
   --------- ------------------------------ 2.6/11.1 MB 1.4 MB/s eta 0:00:07
   ------------ --------------------------- 3.4/11.1 MB 1.7 MB/s eta 0:00:05
   ---------------- ---------------

In [26]:
# get the groups
for prov in provinces:
    process_province(prov)

In [27]:
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import numpy as np

def process_group(group, prov):
    df_group = pd.read_csv(prov + '/' + group + '_2023.csv') 
    # annually heating energy: EGHSPACEENERGY; Furnace efficiency: FURSSEFF
    df_group = df_group[['EGHSPACEENERGY', 'FURSSEFF']]

    # Normalize the Data
    scaler = StandardScaler()
    scaled_df = scaler.fit_transform(df_group)

    # Perform K-Means Clustering
    kmeans = KMeans(n_clusters=10, random_state=42)  
    kmeans.fit(scaled_df)
    df_group['Cluster'] = kmeans.labels_

    # print(df_group)

    # Find Centroids Using Real Data Points
    real_centroids = []

    for cluster in range(kmeans.n_clusters):
        # Get all points in the current cluster
        cluster_points = df_group[df_group['Cluster'] == cluster]
        
        # Get scaled points and their corresponding original indices
        cluster_scaled_points = scaled_df[df_group['Cluster'] == cluster]
        
        # Calculate the Euclidean distances to the calculated centroid
        distances = np.linalg.norm(cluster_scaled_points - kmeans.cluster_centers_[cluster], axis=1)
        
        # Find the indices of the closest real data points
        min_distance_index = np.argsort(distances)[:1]  
        real_centroid = cluster_points.iloc[min_distance_index]
        real_centroids.append(real_centroid)

    # print("Real Centroids (Closest Real Data Points):")
    real_centroids_df = pd.concat(real_centroids)

    # print(real_centroids_df)

    # Calculate Cluster Weights
    total_points = len(df_group)
    cluster_weights = df_group['Cluster'].value_counts(normalize=True).sort_index()

    # print("\nCluster Weights (Proportions):")
    # for cluster, weight in cluster_weights.items():
    #     print(f"Cluster {cluster}: {weight:.2f}")

    # add the cluster proportion to the real_centroids_df
    real_centroids_df['Cluster Proportion'] = cluster_weights.values
    # print(real_centroids_df)

    # save the real_centroids_df to the csv file, if not exists, create the folder
    directory = './cluster_centers' + '/' + prov
    if not os.path.exists(directory):
        os.makedirs(directory)
    real_centroids_df.to_csv(f'./cluster_centers/{prov}/{group}_cluster_centers.csv', index=False)

In [18]:
group = 'ON_Apartment_Electricity_2021_2025'
prov = 'ON'
process_group(group, prov)

FileNotFoundError: [Errno 2] No such file or directory: 'ON/ON_Apartment_Electricity_2021_2025_2023.csv'

In [28]:
# get all groups centers
for prov in provinces:
    directory = './' + prov
    groups = os.listdir(directory)
    for group in groups:

        # only process the group if they have more than 9 data
        df_group = pd.read_csv(directory + '/' + group)
        if len(df_group) < 10:
            continue

        # remove '_2022.csv'
        group = group[:-9]
        process_group(group, prov)

In [21]:
group

'ON_Apartment_Electricity_2021_2025'